In [4]:
# Install required packages (run once)
%pip install -q azure-cosmos -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Module 2: Episodic Memory

Module 1 showed us that session memory is **transient** — lost when the session ends. The obvious fix? Save everything to a database.

In this module we'll try that, discover why it falls apart at scale, and arrive at a better approach: **episodic memory**.

## What You'll Learn
1. How to **persist chat history** to a database using the framework's `HistoryProvider`
2. Why saving every message creates bloat, noise, and unqueryable data
3. How **episodic memory** stores only what matters — structured events the agent can recall
4. The design pattern: `@tool` for selective memory (agent decides what to remember)
5. How persistent chat history and episodic memory **work together** in practice

## The Journey
```
Transient memory (Module 1)
    → Persist every message (naive fix)
        → Discover the scaling problem
            → Episodic memory (selective, structured)
                → Both together (audit + intelligence)
```

---
## Setup

> **Prerequisite**: You need an Azure Cosmos DB account. Follow the steps in
> [`steps/01_setup_cosmos.md`](steps/01_setup_cosmos.md) if you haven't created one yet.
> Then add `COSMOS_ENDPOINT` to your `.env` file.

In [5]:
import os
import json
import asyncio
import nest_asyncio
from datetime import datetime, timedelta, timezone
from uuid import uuid4
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv("../../.env", override=True)

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL", "gpt-4o")
COSMOS_ENDPOINT = os.environ["COSMOS_ENDPOINT"]

print(f"Project:  {PROJECT_ENDPOINT}")
print(f"Model:    {MODEL_DEPLOYMENT}")
print(f"Cosmos:   {COSMOS_ENDPOINT}")

Project:  https://npmsfoundrysc.services.ai.azure.com/api/projects/proj-default
Model:    gpt-5.4-mini
Cosmos:   https://cosmos-agentic-memory-ds.documents.azure.com:443/


In [6]:
from azure.identity import AzureCliCredential
from azure.cosmos.aio import CosmosClient
from azure.cosmos import PartitionKey
from agent_framework import Agent, Message, AgentSession
from agent_framework.foundry import FoundryChatClient
from agent_framework_azure_cosmos import CosmosHistoryProvider

TENANT_ID = os.environ.get("AZURE_TENANT_ID")
credential = AzureCliCredential()

# ─── Create Cosmos DB database and containers ───
# Note: The chat-history container is created automatically by CosmosHistoryProvider
# on first use (partitioned on /session_id). We only need to create episodic-memory here.
async def setup_cosmos():
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        database = await cosmos.create_database_if_not_exists("travel-memory")
        episodic_container = await database.create_container_if_not_exists(
            "episodic-memory",
            partition_key=PartitionKey(path="/user_id"),
        )
        print(f"Database:  travel-memory")
        print(f"Container: episodic-memory    (partition: /user_id)")
        print(f"Container: chat-history       (created automatically by CosmosHistoryProvider)")

asyncio.run(setup_cosmos())

Database:  travel-memory
Container: episodic-memory    (partition: /user_id)
Container: chat-history       (created automatically by CosmosHistoryProvider)


---
## Part 1: The Naive Fix — Persist Every Message

Module 1's problem: session memory vanishes when the session ends.

The natural first idea: **save every chat message to a database**. The Agent Framework makes this easy with `CosmosHistoryProvider` from the `agent-framework-azure-cosmos` package — it automatically stores and reloads conversation history.

```
┌──────────────────────────────────────────┐
│        CosmosHistoryProvider              │
│                                          │
│  before_run()  → loads saved messages    │
│  after_run()   → saves new messages      │
│                                          │
│  Built-in — just configure:              │
│    endpoint, database_name,              │
│    container_name, credential            │
└──────────────────────────────────────────┘
```

Every turn, `after_run` persists the user's input and the agent's response. On the next turn (or a new session with the same ID), `before_run` reloads the full history.

In [7]:
# The official CosmosHistoryProvider handles everything:
# - Creates the container on first use (partition key: /session_id)
# - Loads messages before each agent run (before_run)
# - Saves messages after each agent run (after_run)
# - Uses efficient batch operations for writes

history = CosmosHistoryProvider(
    endpoint=COSMOS_ENDPOINT,
    database_name="travel-memory",
    container_name="chat-history",
    credential=credential,
)
print("CosmosHistoryProvider configured ✓")

CosmosHistoryProvider configured ✓


In [8]:
SYSTEM_INSTRUCTIONS = """You are a corporate travel assistant for Contoso Corp.
Help employees book business travel including flights and hotels.
Be concise and helpful."""

SESSION_ID = "demo-session-1"

async def conversation_with_persistence():
    session = AgentSession(session_id=SESSION_ID)

    agent = Agent(
        client=FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        ),
        name="TravelAssistant",
        instructions=SYSTEM_INSTRUCTIONS,
        context_providers=[history],
    )
    turns = [
        "Hi, I'm Sarah Chen from Engineering.",
        "I need to travel to New York next month for a conference.",
        "I prefer the Marriott and aisle seats on United.",
    ]
    for user_msg in turns:
        print(f"User: {user_msg}")
        response = await agent.run(user_msg, session=session)
        print(f"Agent: {response.text}\n")

    print("\n--- Every message saved to Cosmos DB ---")

asyncio.run(conversation_with_persistence())

User: Hi, I'm Sarah Chen from Engineering.
Agent: Hi Sarah — nice to meet you. How can I help with your business travel today?

User: I need to travel to New York next month for a conference.
Agent: Absolutely — I can help with that.

Please share:
- **Departure city**
- **Travel dates** (or approximate dates next month)
- **Conference location** in New York, if known
- **Any preferences** for flight times, airline, or hotel budget

Once I have that, I can suggest flight and hotel options.

User: I prefer the Marriott and aisle seats on United.
Agent: Got it — Marriott hotel and aisle seats on United.

To find options, I still need:
- **Departure city**
- **Travel dates** next month
- **Conference location** in New York, if known

If you want, I can also keep the hotel near the conference venue and look for United flights with aisle seats.


--- Every message saved to Cosmos DB ---


### Does It Survive a Restart?

The messages are in Cosmos DB. Let's simulate a restart — create a **completely new agent** with the same session ID and see if it remembers.

In [9]:
async def restart_demo():
    """Brand new agent instance, same session ID — does it remember?"""
    session = AgentSession(session_id=SESSION_ID)  # Same session ID

    agent = Agent(
        client=FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        ),
        name="TravelAssistant",
        instructions=SYSTEM_INSTRUCTIONS,
        context_providers=[history],
    )
    # Ask about info from the PREVIOUS session
    print("--- New agent instance, same session ID ---\n")
    user_msg = "What's my name, where am I going, and what are my preferences?"
    print(f"User: {user_msg}")
    response = await agent.run(user_msg, session=session)
    print(f"Agent: {response.text}")

asyncio.run(restart_demo())

--- New agent instance, same session ID ---

User: What's my name, where am I going, and what are my preferences?
Agent: Your name is **Sarah Chen**.

You’re going to **New York** for a **conference next month**.

Your preferences are:
- **Hotel:** Marriott
- **Airline:** United
- **Seat:** aisle seat


### Persisted Chat History Works!

The agent remembered Sarah's name, destination, and preferences — all loaded from Cosmos DB. The `CosmosHistoryProvider` automatically:

1. **`before_run`** → loaded all saved messages for `demo-session-1`
2. **`after_run`** → saved the new user input and agent response

Problem solved? Not quite. Let's see what happens at scale.

---
## Part 2: The Problem with Saving Everything

Our `CosmosHistoryProvider` works perfectly for a few turns. But Sarah is a frequent traveler — she's had dozens of conversations with the assistant over months. Let's simulate that.

In [10]:
# Simulate 6 months of chat history: 20 conversations, ~10 messages each
SYNTHETIC_CONVERSATIONS = [
    ("conv-jan-nyc", "January NYC trip planning", [
        ("user", "I need to book a trip to New York for January 15-18."),
        ("assistant", "I'd be happy to help you plan your NYC trip for January 15-18. Let me look into flights and hotels for you."),
        ("user", "What are the flight options from Seattle?"),
        ("assistant", "I found several flights from Seattle to JFK. United has a direct flight UA101 departing at 8am for $460, and Delta has DL305 at 10am for $420."),
        ("user", "Go with United. What about hotels?"),
        ("assistant", "For hotels, the Marriott Times Square has rooms at $280/night, and the Hilton Midtown is $260/night."),
        ("user", "Marriott please. And can you check if the conference venue is nearby?"),
        ("assistant", "The Marriott Times Square is about a 10-minute walk from the Javits Center. I'll book the Marriott for 3 nights at $280/night."),
        ("user", "Perfect, book it all."),
        ("assistant", "All booked! United UA101 on Jan 15, returning UA103 on Jan 18. Marriott Times Square for 3 nights. Total: $1,300."),
    ]),
    ("conv-feb-budget", "February budget discussion", [
        ("user", "What's my travel budget looking like?"),
        ("assistant", "Looking at your recent trips, you've spent about $3,200 in Q4 last year."),
        ("user", "Is that within policy?"),
        ("assistant", "Yes, Senior Engineers have an annual travel budget of $15,000. You're well within that."),
        ("user", "OK great, thanks for checking."),
        ("assistant", "You're welcome! Let me know if you need anything else."),
    ]),
    ("conv-mar-london", "March London trip planning", [
        ("user", "I need to go to London for a client meeting March 20-25."),
        ("assistant", "I'll look into London flights and hotels for March 20-25."),
        ("user", "Last time the flight was really long. Any better options?"),
        ("assistant", "United has a direct flight from SFO at 6pm, arriving at Heathrow at 11am. About 10.5 hours. British Airways also has a similar route."),
        ("user", "Let's go with United again for the miles."),
        ("assistant", "Good call on the miles. For hotels, there's the Marriott Park Lane at £220/night or the Hilton London at £195/night."),
        ("user", "Marriott. Same chain loyalty points."),
        ("assistant", "Smart thinking. Booking: United direct to LHR March 20, returning March 25. Marriott Park Lane for 5 nights. Total: approximately $3,800."),
        ("user", "Also, do I need a visa?"),
        ("assistant", "US citizens don't need a visa for UK business trips under 6 months. You'll just need your passport."),
        ("user", "Thanks!"),
        ("assistant", "You're welcome! Have a great trip to London."),
    ]),
    ("conv-apr-expense", "April expense report help", [
        ("user", "I need help with my expense report for the London trip."),
        ("assistant", "Of course! I can see your London trip was March 20-25. What do you need help with?"),
        ("user", "What were the exact costs?"),
        ("assistant", "Flight: $2,450 (United round trip). Hotel: $1,450 (Marriott Park Lane, 5 nights). Total: $3,900."),
        ("user", "Were there any meal expenses I should add?"),
        ("assistant", "You had a per diem allowance of $75/day for London, which gives you $375 for 5 days."),
        ("user", "Great, that covers it."),
        ("assistant", "I'll note that down. Your total London trip expense report should be $4,275."),
    ]),
    ("conv-may-cancel", "May trip cancellation", [
        ("user", "I need to cancel my Chicago trip next week."),
        ("assistant", "I see you have a Chicago trip booked for next week. I'll start the cancellation process. What's the reason?"),
        ("user", "The meeting got moved to a video call instead."),
        ("assistant", "Understood. I'll cancel the United flight and Hilton reservation. The flight has a cancellation fee of $150."),
        ("user", "That's fine, go ahead."),
        ("assistant", "Done. Flight and hotel cancelled. $150 cancellation fee applied. You'll get a credit for the hotel."),
    ]),
    ("conv-jun-team", "June team offsite discussion", [
        ("user", "We're planning a team offsite in Austin. 8 people, June 15-17."),
        ("assistant", "A team offsite in Austin sounds great! For 8 people over June 15-17, I'll look into group rates."),
        ("user", "We need a hotel with a meeting room."),
        ("assistant", "The Hilton Austin has group rates at $189/person/night and includes a meeting room for groups of 8+."),
        ("user", "What about flights for everyone from Seattle?"),
        ("assistant", "Southwest has the best group rates from Seattle to Austin. About $320 round trip per person."),
        ("user", "Can you send me a summary I can share with the team?"),
        ("assistant", "Here's the summary: Austin offsite, June 15-17. Hilton Austin at $189/night/person. Southwest flights at ~$320/person. Total estimate: ~$5,600 for 8 people."),
        ("user", "Looks good. I'll get budget approval and come back."),
        ("assistant", "Sounds like a plan! Let me know once you have approval and I'll make the bookings."),
    ]),
]

async def simulate_months_of_history():
    """Write synthetic conversations to Cosmos DB to simulate scale."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        container = cosmos.get_database_client("travel-memory").get_container_client("chat-history")

        total_messages = 0
        for conv_id, description, messages in SYNTHETIC_CONVERSATIONS:
            for i, (role, content) in enumerate(messages):
                doc = {
                    "id": f"{conv_id}-{i:03d}",
                    "session_id": conv_id,
                    "sort_key": i,
                    "source_id": "azure_cosmos_history",
                    "message": Message("user" if role == "user" else "assistant", [content]).to_dict(),
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                }
                await container.upsert_item(doc)
                total_messages += 1
        print(f"Loaded {len(SYNTHETIC_CONVERSATIONS)} conversations, {total_messages} messages")

asyncio.run(simulate_months_of_history())

Loaded 6 conversations, 52 messages


In [11]:
async def show_the_bloat():
    """Count ALL messages across ALL conversations in chat-history."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        container = cosmos.get_database_client("travel-memory").get_container_client("chat-history")

        # Count everything
        total = 0
        conversations = set()
        total_chars = 0
        async for item in container.read_all_items():
            total += 1
            conversations.add(item["session_id"])
            if "message" in item:
                msg = Message.from_dict(item["message"])
                total_chars += len(msg.text)

        estimated_tokens = total_chars // 4  # rough estimate

        print(f"Total messages in chat-history:  {total}")
        print(f"Across conversations:            {len(conversations)}")
        print(f"Total characters:                {total_chars:,}")
        print(f"Estimated tokens:                ~{estimated_tokens:,}")
        print()

        # Now try to answer a simple question from this data
        print("--- Try to find: 'What hotel did Sarah like in NYC?' ---")
        print()
        nyc_hits = 0
        async for item in container.query_items(
            query="SELECT c.session_id, c.message FROM c WHERE CONTAINS(LOWER(c.message.contents[0].text), 'marriott')",
        ):
            nyc_hits += 1
            msg = Message.from_dict(item["message"])
            print(f"  [{item['session_id']}] {msg.role}: {msg.text[:80]}...")
        print(f"\n  Found {nyc_hits} messages mentioning 'marriott' — scattered across conversations.")
        print("  No structure, no ratings, no way to rank which trip mattered most.")

asyncio.run(show_the_bloat())

Total messages in chat-history:  60
Across conversations:            7
Total characters:                4,641
Estimated tokens:                ~1,160

--- Try to find: 'What hotel did Sarah like in NYC?' ---

  [demo-session-1] user: I prefer the Marriott and aisle seats on United....
  [demo-session-1] assistant: Got it — Marriott hotel and aisle seats on United.

To find options, I still nee...
  [demo-session-1] assistant: Your name is **Sarah Chen**.

You’re going to **New York** for a **conference ne...
  [conv-jan-nyc] assistant: For hotels, the Marriott Times Square has rooms at $280/night, and the Hilton Mi...
  [conv-jan-nyc] user: Marriott please. And can you check if the conference venue is nearby?...
  [conv-jan-nyc] assistant: The Marriott Times Square is about a 10-minute walk from the Javits Center. I'll...
  [conv-jan-nyc] assistant: All booked! United UA101 on Jan 15, returning UA103 on Jan 18. Marriott Times Sq...
  [conv-mar-london] assistant: Good call on the miles.

### Three Problems with Persisted Chat History

| Problem | Description |
|---------|-------------|
| **Bloat** | Hundreds of messages for information that could fit in a few structured records |
| **Noise** | Small talk, corrections, tangents — all stored with equal weight |
| **Unqueryable** | "What hotel did she like in NYC?" requires searching raw text across conversations |

Persisted chat history is great for **audit trails** — you can always replay what was said. But it's a terrible way to build **intelligence**. The agent can't efficiently extract meaning from a wall of raw messages.

### "But What About Vector Search?"

You might think: *"Just add embeddings and do vector search over the messages."* Cosmos DB even supports this natively. But vector search only addresses one of the three problems:

| Problem | Does vector search help? |
|---------|---------------------------|
| **Bloat** | ❌ No — you still store and embed every message. The data keeps growing. |
| **Noise** | ❌ No — "sure, book it" and "thanks!" get embedded alongside useful content |
| **Unqueryable** | ✅ Partly — semantic matching helps, but returns raw text, not structured fields. You can't filter by rating > 4 or sort by cost. |

Vector search is a retrieval technique, not a memory architecture. It helps you *find* messages, but doesn't help you *understand* them. You still end up injecting raw conversation fragments into the agent's context.

### Context Is a Scarce Resource

Every LLM has a finite context window. Every token you spend on noise is a token you can't spend on useful context — the user's current request, relevant policies, structured data.

> **Rule of thumb**: Be deliberate about what you load into an agent's context. Context window tokens are a scarce resource — fill them with signal, not noise.

### If This Sounds Familiar…

This is the same trade-off as **virtual memory paging** in an operating system:

| OS Paging | Agent Memory |
|-----------|-------------|
| **RAM** (fast, limited) | Last N messages in the context window |
| **Disk** (slow, large) | Episodic memory in the database |
| **Page fault** → load from disk | Agent calls `recall_events` when it needs older context |
| **LRU eviction** | Oldest messages drop out of the sliding window |

Keep the most recent conversation in the "fast" context window. When the agent needs something older — "what hotel did I like in London?" — it "page faults" by calling a tool, which pulls structured data from the database on demand. Same principle, different layer of the stack.

### What If We Could Be Selective?

Instead of saving every message, what if the agent could **decide what's worth remembering** and store it in a structured way?

```
Chat History (saves everything):          Episodic Memory (saves what matters):
─────────────────────────────             ─────────────────────────────────────
"Hi, I'm Sarah"                           Event: NYC trip, Oct 2025
"Hello Sarah!"                              Hotel: Marriott Times Square
"I need to go to NYC"                       Rating: ★★★★★
"Sure, when?"                               Note: "Great location, close to venue"
"October 15-18"
"Let me check flights..."                 Event: London trip, Nov 2025
"United has UA101 at 8am"                   Hotel: Marriott Park Lane
"Book it"                                   Rating: ★★★★
"What about hotels?"                        Note: "Long flight but productive"
"Marriott is $280/night"
"Book it"
"All set!"
...12 messages                            ...2 records
```

This is **episodic memory** — storing curated events instead of raw conversation.

---
## Part 3: Episodic Memory — Remembering What Matters

**Episodic memory** stores specific events and experiences as structured records. It answers questions like:
- "What happened last time I went to NYC?"
- "Which hotels have I stayed at?"
- "How was my London trip?"

The key difference from chat history: the **agent decides** what's worth remembering and stores it as a searchable event — not as raw conversation text.

### The Pattern: `@tool`

We give the agent two tools:
- **`remember_event`** — store a structured event (agent calls this when something significant happens)
- **`recall_events`** — search past events (agent calls this when it needs historical context)

The agent uses judgment about **when** to call these tools. This selectivity is the whole point.

In [12]:
from agent_framework import tool

# We'll use a module-level container reference for the tools
_episodic_container = None

def set_episodic_container(container):
    global _episodic_container
    _episodic_container = container

@tool
async def remember_event(
    user_id: str,
    event_type: str,
    description: str,
    details: str = "",
) -> str:
    """Store a significant event in the user's episodic memory.

    Call this when:
    - A trip is completed or booked
    - The user shares feedback about a past experience
    - Something notable happens worth remembering for the future

    Args:
        user_id: Employee ID (e.g. "E001")
        event_type: Category — "trip", "booking", "feedback", "preference"
        description: Brief summary of what happened
        details: Optional JSON string with structured data (dates, costs, ratings)
    """
    event = {
        "id": f"{user_id}-{uuid4().hex[:8]}",
        "user_id": user_id,
        "event_type": event_type,
        "description": description,
        "details": json.loads(details) if details else {},
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    await _episodic_container.upsert_item(event)
    return f"Remembered: {description}"


@tool
async def recall_events(
    user_id: str,
    query: str = "",
    event_type: str = "",
    limit: int = 5,
) -> str:
    """Search the user's episodic memory for past events.

    Call this when:
    - Planning a trip and want to check past experiences
    - The user asks about their travel history
    - Need context from previous trips for recommendations

    Args:
        user_id: Employee ID (e.g. "E001")
        query: Optional keyword to search in descriptions
        event_type: Optional filter — "trip", "booking", "feedback"
        limit: Maximum number of events to return
    """
    sql = "SELECT * FROM c WHERE c.user_id = @uid"
    params = [{"name": "@uid", "value": user_id}]

    if event_type:
        sql += " AND c.event_type = @etype"
        params.append({"name": "@etype", "value": event_type})
    if query:
        sql += " AND CONTAINS(LOWER(c.description), @q)"
        params.append({"name": "@q", "value": query.lower()})

    sql += " ORDER BY c.timestamp DESC"

    events = []
    async for item in _episodic_container.query_items(
        query=sql, parameters=params, partition_key=user_id
    ):
        events.append({
            "event_type": item["event_type"],
            "description": item["description"],
            "details": item.get("details", {}),
            "timestamp": item["timestamp"],
        })
        if len(events) >= limit:
            break

    return json.dumps(events, indent=2) if events else "No matching events found."

print("Episodic memory tools defined: remember_event, recall_events ✓")

Episodic memory tools defined: remember_event, recall_events ✓


In [13]:
async def preload_episodic_events():
    """Load past trips from data file as episodic memory events."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        container = cosmos.get_database_client("travel-memory").get_container_client("episodic-memory")

        with open("../../data/past_trips.json") as f:
            trips = json.load(f)

        for trip in trips:
            event = {
                "id": f"trip-{trip['trip_id']}",
                "user_id": trip["employee_id"],
                "event_type": "trip",
                "description": (
                    f"{trip['destination']} trip for {trip['purpose']} "
                    f"({trip['departure_date']} to {trip['return_date']})"
                ),
                "details": {
                    "destination": trip["destination"],
                    "purpose": trip["purpose"],
                    "dates": f"{trip['departure_date']} to {trip['return_date']}",
                    "hotel_id": trip["hotel"]["hotel_id"],
                    "total_cost": trip["total_cost"],
                    "rating": trip["rating"],
                    "notes": trip["notes"],
                },
                "timestamp": trip["departure_date"] + "T00:00:00",
            }
            await container.upsert_item(event)
            print(f"  Loaded: {event['description']}")

    print(f"\nLoaded {len(trips)} trips as episodic events")

asyncio.run(preload_episodic_events())

  Loaded: New York trip for Q4 Engineering Summit (2025-10-15 to 2025-10-18)
  Loaded: London trip for Client meeting with Acme Corp (2025-11-20 to 2025-11-25)
  Loaded: New York trip for Board presentation (2025-12-01 to 2025-12-02)
  Loaded: Tokyo trip for Partner conference (2026-01-10 to 2026-01-15)
  Loaded: New York trip for Team offsite (2026-02-20 to 2026-02-22)

Loaded 5 trips as episodic events


In [21]:
EPISODIC_INSTRUCTIONS = """You are a corporate travel assistant for Contoso Corp.
Help employees book business travel including flights and hotels.

You have access to the employee's episodic memory — their past trips and experiences.
When planning new trips:
1. Use recall_events to check if they've been to this destination before
2. Reference past experiences to make better recommendations
3. Use remember_event to store significant outcomes after bookings

The current user is Sarah Chen (employee ID: E001, based in Seattle, USA).
Be concise and helpful."""

async def demo_episodic_recall():
    """Show the agent using episodic memory to recall past trips."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        container = cosmos.get_database_client("travel-memory").get_container_client("episodic-memory")
        set_episodic_container(container)

        agent = Agent(
            client=FoundryChatClient(
                project_endpoint=PROJECT_ENDPOINT,
                model=MODEL_DEPLOYMENT,
                credential=credential,
            ),
            name="TravelAssistant",
            instructions=EPISODIC_INSTRUCTIONS,
            tools=[remember_event, recall_events],
        )
        user_msg = "I'm planning another trip to New York. What do you know about my past New York trips?"
        print(f"User: {user_msg}\n")
        response = await agent.run(user_msg)
        print(f"Agent: {response.text}")

asyncio.run(demo_episodic_recall())

User: I'm planning another trip to New York. What do you know about my past New York trips?

Agent: You’ve been to New York twice that I can see:

- **2026-02-20 to 2026-02-22** — Team offsite  
  - Hotel: **H001**
  - Total cost: **$1,420**
  - Rating: **4/5**
  - Note: “Same hotel as before, still good”

- **2025-10-15 to 2025-10-18** — Q4 Engineering Summit  
  - Hotel: **H001**
  - Total cost: **$1,760**
  - Rating: **5/5**
  - Note: “Great hotel location, close to venue”

Pattern-wise, you seem to have liked **hotel H001** in New York, especially for location. If you want, I can use that history to suggest similar hotel options for your next trip.


### What Just Happened

The agent:
1. Received the user's question about NYC
2. **Decided** to call `recall_events(user_id="E001", query="new york")` — its own judgment
3. Retrieved structured records: destinations, ratings, costs, notes
4. Used that context to give a personalized, informed response

Compare this to searching through hundreds of raw chat messages. The episodic events are:
- **Structured** — fields like `rating`, `hotel_id`, `total_cost`
- **Queryable** — filter by destination, event type, date range
- **Compact** — one record per trip instead of dozens of messages

In [15]:
async def demo_episodic_store():
    """Show the agent storing a new episodic event."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        container = cosmos.get_database_client("travel-memory").get_container_client("episodic-memory")
        set_episodic_container(container)

        agent = Agent(
            client=FoundryChatClient(
                project_endpoint=PROJECT_ENDPOINT,
                model=MODEL_DEPLOYMENT,
                credential=credential,
            ),
            name="TravelAssistant",
            instructions=EPISODIC_INSTRUCTIONS,
            tools=[remember_event, recall_events],
        )
        user_msg = (
            "I just got back from a conference in Chicago. The Palmer House Hilton was amazing — "
            "5 stars. Great location, walking distance to the venue. Total cost was $1,850."
        )
        print(f"User: {user_msg}\n")
        response = await agent.run(user_msg)
        print(f"Agent: {response.text}")

asyncio.run(demo_episodic_store())

User: I just got back from a conference in Chicago. The Palmer House Hilton was amazing — 5 stars. Great location, walking distance to the venue. Total cost was $1,850.

Agent: Glad to hear it went well — I’ve saved your feedback about The Palmer House Hilton in Chicago.


In [16]:
async def verify_stored_event():
    """Check what the agent stored in episodic memory."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        container = cosmos.get_database_client("travel-memory").get_container_client("episodic-memory")

        print("--- Sarah's episodic memory (most recent first) ---\n")
        events = []
        async for item in container.query_items(
            query="SELECT * FROM c WHERE c.user_id = 'E001' ORDER BY c.timestamp DESC",
            partition_key="E001",
        ):
            events.append(item)

        for e in events:
            details = e.get("details", {})
            rating = details.get("rating", "—")
            cost = details.get("total_cost", "—")
            print(f"  [{e['event_type']}] {e['description']}")
            print(f"           Rating: {rating}  Cost: ${cost}")
            print()

        print(f"Total episodic events for Sarah: {len(events)}")

asyncio.run(verify_stored_event())

--- Sarah's episodic memory (most recent first) ---

  [feedback] User shared positive feedback about a recent stay at The Palmer House Hilton in Chicago for a conference.
           Rating: 5  Cost: $1850

  [trip] New York trip for Team offsite (2026-02-20 to 2026-02-22)
           Rating: 4  Cost: $1420

  [trip] London trip for Client meeting with Acme Corp (2025-11-20 to 2025-11-25)
           Rating: 4  Cost: $3900

  [trip] New York trip for Q4 Engineering Summit (2025-10-15 to 2025-10-18)
           Rating: 5  Cost: $1760

Total episodic events for Sarah: 4


In [ ]:
async def compare_approaches():
    """Compare chat history vs episodic memory for answering the same question."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        db = cosmos.get_database_client("travel-memory")
        chat_container = db.get_container_client("chat-history")
        episodic_container = db.get_container_client("episodic-memory")

        # Chat history approach: search raw messages
        print("═" * 60)
        print("APPROACH 1: Search Chat History for 'New York hotel'")
        print("═" * 60)
        chat_hits = 0
        chat_chars = 0
        async for item in chat_container.query_items(
            query=("SELECT * FROM c WHERE "
                   "CONTAINS(LOWER(c.message.contents[0].text), 'marriott') OR "
                   "CONTAINS(LOWER(c.message.contents[0].text), 'new york') OR "
                   "CONTAINS(LOWER(c.message.contents[0].text), 'nyc')"),
        ):
            chat_hits += 1
            msg = Message.from_dict(item["message"])
            chat_chars += len(msg.text)

        print(f"  Messages found:  {chat_hits}")
        print(f"  Characters:      {chat_chars:,}")
        print(f"  Tokens (est):    ~{chat_chars // 4:,}")
        print(f"  Structured?      No — raw conversation text")
        print(f"  Has ratings?     No — buried in natural language")

        # Episodic approach: query structured events
        print()
        print("═" * 60)
        print("APPROACH 2: Query Episodic Memory for New York trips")
        print("═" * 60)
        ep_hits = 0
        ep_chars = 0
        async for item in episodic_container.query_items(
            query="SELECT * FROM c WHERE c.user_id = 'E001' AND CONTAINS(LOWER(c.description), 'new york')",
            partition_key="E001",
        ):
            ep_hits += 1
            ep_chars += len(json.dumps(item.get("details", {})))
            details = item.get("details", {})
            print(f"  → {item['description']}")
            print(f"    Rating: {details.get('rating', '—')}  Cost: ${details.get('total_cost', '—')}")

        print(f"\n  Events found:    {ep_hits}")
        print(f"  Characters:      {ep_chars:,}")
        print(f"  Tokens (est):    ~{ep_chars // 4:,}")
        print(f"  Structured?      Yes — typed fields, queryable")
        print(f"  Has ratings?     Yes — directly accessible")

asyncio.run(compare_approaches())

════════════════════════════════════════════════════════════
APPROACH 1: Search Chat History for 'NYC hotel'
════════════════════════════════════════════════════════════
  Messages found:  15
  Characters:      1,826
  Tokens (est):    ~456
  Structured?      No — raw conversation text
  Has ratings?     No — buried in natural language

════════════════════════════════════════════════════════════
APPROACH 2: Query Episodic Memory for NYC trips
════════════════════════════════════════════════════════════
  → New York trip for Q4 Engineering Summit (2025-10-15 to 2025-10-18)
    Rating: 5  Cost: $1760
  → New York trip for Team offsite (2026-02-20 to 2026-02-22)
    Rating: 4  Cost: $1420

  Events found:    2
  Characters:      391
  Tokens (est):    ~97
  Structured?      Yes — typed fields, queryable
  Has ratings?     Yes — directly accessible


---
## Part 4: Better Together

We've seen two approaches in isolation. In production, you'll typically use **both**:

- **`HistoryProvider`** → automatic conversation persistence (audit trail, continuity)
- **`@tool` episodic memory** → selective, structured knowledge (intelligence)

They serve different roles and complement each other. Let's see them working side by side.

In [25]:
COMBINED_SESSION_ID = "demo-combined-1"

async def demo_combined_memory():
    """Agent with BOTH persistent history AND episodic memory tools."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        db = cosmos.get_database_client("travel-memory")
        episodic_container = db.get_container_client("episodic-memory")
        set_episodic_container(episodic_container)

        session = AgentSession(session_id=COMBINED_SESSION_ID)

        agent = Agent(
            client=FoundryChatClient(
                project_endpoint=PROJECT_ENDPOINT,
                model=MODEL_DEPLOYMENT,
                credential=credential,
            ),
            name="TravelAssistant",
            instructions=EPISODIC_INSTRUCTIONS,
            tools=[remember_event, recall_events],
            context_providers=[history],
        )
        turns = [
            "Hi, I'm Sarah Chen (E001). I need to plan a trip to Tokyo next month 15th June.",
            "Can you check my past trips for any Asia travel experience?",
            "Book the ANA flight and the Tokyo Marriott please.",
        ]
        for user_msg in turns:
            print(f"User: {user_msg}")
            response = await agent.run(user_msg, session=session)
            print(f"Agent: {response.text}\n")

        print("--- Conversation persisted to chat-history, episodic memory available throughout ---")

asyncio.run(demo_combined_memory())

User: Hi, I'm Sarah Chen (E001). I need to plan a trip to Tokyo next month 15th June.
Agent: Great — Tokyo next month on **15 June**.

I’m missing a few details to plan it properly:
- **Return date**
- **Departure city** (I have Seattle on file—please confirm)
- **Cabin class** for ANA
- **Hotel budget**
- **Preferred area in Tokyo** or whether you want me to suggest one

Once you share those, I can help build the trip.

User: Can you check my past trips for any Asia travel experience?
Agent: I checked your past trips for Asia travel and found no matching records.

If that’s surprising, it may mean:
- no Asia trips are on file yet, or
- the trips weren’t categorized in a way I can search right now

If you want, send me your Tokyo dates and I’ll help plan from scratch.

User: Book the ANA flight and the Tokyo Marriott please.
Agent: I can help, but I still need the trip details to book them:

- exact departure and return dates
- preferred flight cabin class
- hotel dates / number of nig

In [27]:
async def demo_combined_resume():
    """New agent, same session — needs BOTH memory systems to answer."""
    async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
        db = cosmos.get_database_client("travel-memory")
        episodic_container = db.get_container_client("episodic-memory")
        set_episodic_container(episodic_container)

        session = AgentSession(session_id=COMBINED_SESSION_ID)

        agent = Agent(
            client=FoundryChatClient(
                project_endpoint=PROJECT_ENDPOINT,
                model=MODEL_DEPLOYMENT,
                credential=credential,
            ),
            name="TravelAssistant",
            instructions=EPISODIC_INSTRUCTIONS,
            tools=[remember_event, recall_events],
            context_providers=[history],
        )
        print("--- New agent, same session ID ---\n")
        user_msg = (
            "What did we discuss about my trip to Tokyo, "
            "and how was my last New York trip?"
        )
        print(f"User: {user_msg}\n")
        response = await agent.run(user_msg, session=session)
        print(f"Agent: {response.text}")

asyncio.run(demo_combined_resume())

--- New agent, same session ID ---

User: What did we discuss about my trip to Tokyo, and how was my last New York trip?

Agent: For Tokyo, we discussed that you want to plan a trip next month and you mentioned **15 June** as the departure date. We also talked about **ANA** for the flight and **Tokyo Marriott** for the hotel, but no booking is confirmed yet because key details are still missing:
- return date
- departure city
- cabin class
- hotel dates / number of nights
- budget or preferred Tokyo area

For your last New York trip:
- **Trip:** Team offsite
- **Dates:** 20–22 Feb 2026
- **Hotel:** same one as before
- **Rating:** 4/5
- **Note:** “Same hotel as before, still good”
- **Cost:** $1,420

If you want, I can help turn the Tokyo plan into a booking-ready itinerary.


### Two Memory Systems, One Agent

The agent answered using **both** systems:

| Question | Source | How |
|----------|--------|-----|
| "What did we discuss about my trip to Tokyo?" | **Chat history** | `CosmosHistoryProvider` reloaded the previous conversation |
| "How was my last New York trip?" | **Episodic memory** | Agent called `recall_events(user_id="E001", query="new york")` |

```
┌──────────────────────────────────────────────────────────┐
│                    Travel Assistant                       │
│                                                          │
│  context_providers=[history]    tools=[remember, recall] │
│          │                              │                │
│          ▼                              ▼                │
│  ┌──────────────┐              ┌─────────────────┐       │
│  │ Chat History  │              │ Episodic Memory │       │
│  │ (automatic)   │              │ (selective)     │       │
│  │  every turn   │              │  curated events │       │
│  │  raw messages │              │  structured     │       │
│  └──────────────┘              └─────────────────┘       │
│         │                              │                 │
│         ▼                              ▼                 │
│     Cosmos DB                     Cosmos DB              │
│   chat-history                  episodic-memory          │
└──────────────────────────────────────────────────────────┘
```

The `CosmosHistoryProvider` gives **continuity** — the agent can resume conversations. The `@tool` episodic memory gives **intelligence** — the agent can learn from the past.

---
## Summary

### Two Approaches to Persistence

| | Chat History | Episodic Memory |
|---|---|---|
| **What's stored** | Every message, every turn | Curated events and outcomes |
| **Who decides** | Automatic (framework) | Agent judgment (`@tool`) |
| **Structure** | Raw text | Typed fields (rating, cost, dates) |
| **Queryable** | Text search only | Filter by type, destination, date |
| **Scales** | Grows with every conversation | Grows with significant events |
| **Best for** | Audit trails, compliance | Intelligence, recommendations |

### Key Insights

1. **Chat history is for audit, episodic memory is for intelligence.** Persisted conversations tell you what was *said*. Episodic events tell you what *happened* and what it *means*.

2. **Context is a scarce resource.** Every LLM has a finite context window. Be deliberate about what you load — fill it with signal, not noise.

3. **Use both.** A `CosmosHistoryProvider` for continuity and compliance. `@tool` episodic memory for the agent's working knowledge. They complement each other.

---
## What's Next

Episodic memory captures **events** — things that happened. But what about **facts**?

- "Sarah prefers aisle seats" isn't an event — it's a persistent fact.
- "Sarah's manager is Michael Torres" is a relationship, not an episode.

In **Module 3: Semantic Memory**, we'll add a memory system for facts, preferences, and relationships — the kind of knowledge that's always true (until it changes).

In [20]:
# Optional: Clean up Cosmos DB containers
# Uncomment and run to delete all data created in this module

# async def cleanup():
#     async with CosmosClient(COSMOS_ENDPOINT, credential=credential) as cosmos:
#         db = cosmos.get_database_client("travel-memory")
#         # Delete containers
#         await db.delete_container("chat-history")
#         await db.delete_container("episodic-memory")
#         print("Cleaned up: chat-history and episodic-memory containers deleted")

# asyncio.run(cleanup())